#  LangChain의 RAG 콤포넌트 - 문서 임베딩(Embeddings) 

### **학습 목표:**  임베딩 모델과 벡터 데이터베이스를 효과적으로 연동할 수 있다

### **실습 자료**: 

- data/transformer.pdf

---

# 환경 설정 및 준비

`(1) Env 환경변수`

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [3]:
import os
from glob import glob  

from pprint import pprint  
import json

`(3) 문서 로드`

In [4]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 로더 초기화
pdf_loader = PyPDFLoader('./data/transformer.pdf')

# 동기 로딩
pdf_docs = pdf_loader.load()
print(f'PDF 문서 개수: {len(pdf_docs)}')

C:\Users\JSPark\AppData\Local\Temp\ipykernel_31120\3020463323.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
e:\sw\dev\ai\modu_llm7\faq_bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PDF 문서 개수: 15


`(4) 텍스트 분할`

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 초기화
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,             # 청크 크기  
    chunk_overlap=200,           # 청크 중 중복되는 부분 크기
    length_function=len,         # 글자 수를 기준으로 분할
    separators=["\n\n", "\n", " ", ""],  # 구분자 - 재귀적으로 순차적으로 적용 
)

# PDF 문서를 텍스트로 분할
chunks = text_splitter.split_documents(pdf_docs)
print(f"생성된 텍스트 청크 수: {len(chunks)}")
print(f"각 청크의 길이: {list(len(chunk.page_content) for chunk in chunks)}")

생성된 텍스트 청크 수: 52
각 청크의 길이: [986, 910, 975, 452, 933, 995, 902, 907, 996, 385, 924, 954, 216, 924, 901, 950, 995, 913, 908, 870, 945, 973, 946, 997, 196, 980, 980, 946, 938, 999, 943, 920, 734, 958, 946, 945, 617, 983, 988, 994, 624, 944, 909, 941, 914, 986, 925, 927, 847, 812, 815, 818]


# 문서 임베딩(Document Embedding)

- 개념: 
    - 텍스트를 벡터(숫자 배열)로 변환하는 과정
    - 문서의 의미적 특성을 수치화하여 컴퓨터가 이해하고 처리할 수 있는 형태로 변환 

- 목적:
    - 텍스트 간 유사도 계산 가능
    - 벡터 데이터베이스 저장 및 검색
    - 의미 기반 문서 검색 구현

- LangChain의 임베딩 모델 종류:
    - OpenAI 임베딩
    - HuggingFace 임베딩 
    - Ollama 임베딩

### 1. **OpenAI**

- LangChain에서 가장 널리 사용되는 임베딩 모델 중 하나

- 주요 특징:
    1. 고품질의 임베딩 생성
    2. 다양한 언어 지원 (다국어 지원)
    3. 일관된 성능
    4. 손쉬운 통합

- 사용시 주의사항:
    1. API 키 설정이 필요 (환경 변수 OPENAI_API_KEY)
    2. API 사용량에 따른 비용 발생
    3. 긴 텍스트는 자동으로 분할되지 않으므로 필요시 TextSplitter를 사용


- 모델별 특징

    | 모델명 | 가격 효율 (페이지/1달러) | 성능 (MTEB) | 최대 입력 | 기본 차원 | 특징 및 추천 |
    | :--- | :---: | :---: | :---: | :---: | :--- |
    | **`text-embedding-3-small`** | **62,500** | 62.3% | 8,191 | 1,536 | **[가성비]** ada-002 대비 5배 저렴하고 성능 우수. 일반적인 RAG 구축 시 1순위. |
    | **`text-embedding-3-large`** | 9,615 | **64.6%** | 8,191 | 3,072 | **[고성능]** 미세한 의미 차이 구분이 중요할 때 사용. 차원 축소 기능 지원. |
    | **`text-embedding-ada-002`** | 12,500 | 61.0% | 8,191 | 1,536 | **[레거시]** 기존에 널리 쓰이던 모델이나, 현재는 v3-small 사용을 권장함. |

> **💡 Tip:** `text-embedding-3` 계열은 **Matryoshka Embedding** 기술이 적용되어, 저장 공간 절약을 위해 임베딩 차원(예: 1536 → 512)을 줄여서 요청해도 성능 하락이 매우 적습니다.

`(1) embedding 모델`

In [6]:
from langchain_openai import OpenAIEmbeddings

# OpenAIEmbeddings 모델 생성
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-large",  # 사용할 모델 이름
    dimensions=None, # 원하는 임베딩 차원 수를 지정 가능 (기본값: None)
    )

# 임베딩 객체 출력
embeddings_model

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000002D8B0E9D130>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000002D8B111D580>, model='text-embedding-3-large', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [7]:
# 임베딩 모델의 컨텍스트 길이 확인
embeddings_model.embedding_ctx_length

8191

In [8]:
# 임베딩 모델의 임베딩 차원 확인 - 기본값 (None)
embeddings_model.dimensions

In [9]:
# OpenAIEmbeddings 모델 생성할 때 임베딩 차원을 지정하는 예시
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",  # 사용할 모델 이름
    dimensions=512, # 원하는 임베딩 차원 수를 지정 가능 (기본값: None)
    )

# 임베딩 모델의 임베딩 차원 확인 
embeddings_model.dimensions

512

In [10]:
# OpenAIEmbeddings 모델 생성
embeddings_openai = OpenAIEmbeddings(model="text-embedding-3-small")

# 임베딩 객체 출력
embeddings_openai

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000002D8B1A08BF0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000002D8B1A68F20>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

`(2) embed_documents 사용`

In [11]:
# 문서 컬렉션
documents = [
    "인공지능은 컴퓨터 과학의 한 분야입니다.",
    "머신러닝은 인공지능의 하위 분야입니다.",
    "딥러닝은 머신러닝의 한 종류입니다.",
    "자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.",
    "컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다."
]

# 문서 임베딩
document_embeddings_openai = embeddings_openai.embed_documents(documents)

# 임베딩 결과 출력
print(f"임베딩 벡터의 개수: {len(document_embeddings_openai)}")
print(f"임베딩 벡터의 차원: {len(document_embeddings_openai[0])}")
print(document_embeddings_openai[0])

임베딩 벡터의 개수: 5
임베딩 벡터의 차원: 1536
[-0.00223541259765625, 0.0121307373046875, -0.002475738525390625, 0.01500701904296875, 0.0184478759765625, -0.045654296875, -0.0031337738037109375, 0.04693603515625, -0.0182342529296875, -0.031829833984375, 0.006900787353515625, 0.006969451904296875, -0.018524169921875, -0.027069091796875, 0.0050201416015625, -0.0155792236328125, -0.041595458984375, 0.01004791259765625, 0.051544189453125, -0.04583740234375, -0.0146636962890625, -0.027801513671875, -0.01873779296875, -0.0226898193359375, 0.004425048828125, -0.03973388671875, 0.05029296875, 0.0177001953125, 0.0002765655517578125, -0.0233154296875, 0.04888916015625, -0.01523590087890625, -0.0299224853515625, -0.0616455078125, 0.020477294921875, 0.035400390625, 0.0026092529296875, -0.00688934326171875, -0.005123138427734375, 0.0247039794921875, 0.007549285888671875, 0.027557373046875, -0.002521514892578125, 0.0260772705078125, -0.0179443359375, 0.00543975830078125, -0.0258026123046875, 0.003643035888671875, 0

`(3) embed_query 사용`

In [12]:
embedded_query_openai = embeddings_openai.embed_query("인공지능이란 무엇인가요?")

# 쿼리 임베딩 결과 출력
print(f"쿼리 임베딩 벡터의 차원: {len(embedded_query_openai)}")
print(embedded_query_openai)

쿼리 임베딩 벡터의 차원: 1536
[-0.0224456787109375, 0.0222015380859375, 0.0003859996795654297, 0.00574493408203125, 0.0122528076171875, -0.044769287109375, -0.026275634765625, 0.035797119140625, -0.0025959014892578125, 0.01476287841796875, -0.0019893646240234375, 0.0009360313415527344, -0.004917144775390625, -0.0736083984375, 0.00899505615234375, -0.0154266357421875, -0.05987548828125, -0.02239990234375, 0.022491455078125, -0.06915283203125, -0.02923583984375, 0.023223876953125, -0.036865234375, 0.0041046142578125, 0.011077880859375, -0.052734375, 0.01071929931640625, 9.28044319152832e-05, -0.0108489990234375, -0.03424072265625, 0.0253143310546875, -0.018829345703125, -0.0015420913696289062, -0.058807373046875, 0.0498046875, -0.0034999847412109375, -0.0028629302978515625, -0.00855255126953125, -0.00460052490234375, 0.0228729248046875, -0.01197052001953125, 0.0379638671875, 0.0033168792724609375, 0.035400390625, -0.051361083984375, 0.034393310546875, -0.0285491943359375, 0.004730224609375, -0.013

`(4) 유사도 기반 검색`

In [13]:
from langchain_community.utils.math import cosine_similarity
import numpy as np

# 쿼리와 가장 유사한 문서 찾기 함수
def find_most_similar(
        query: str, 
        doc_embeddings: np.ndarray,
        embeddings_model  # 기본값 제거, 명시적 전달 강제
        ) -> tuple[str, float]:
    """
    쿼리와 가장 유사한 문서를 찾는 함수
    
    Args:
        query: 검색 쿼리 문자열
        doc_embeddings: 문서 임베딩 배열
        embeddings_model: 임베딩 모델 객체
    
    Returns:
        tuple: (가장 유사한 문서, 유사도 점수)
    """
    # 쿼리 임베딩: OpenAI 임베딩 사용 
    query_embedding = embeddings_model.embed_query(query)

    # 코사인 유사도 계산
    similarities = cosine_similarity([query_embedding], doc_embeddings)[0]

    # 가장 유사한 문서 인덱스 찾기
    most_similar_idx = np.argmax(similarities)

    # 가장 유사한 문서와 유사도 반환: 문서, 유사도
    return documents[most_similar_idx], similarities[most_similar_idx]

# 예제 쿼리
queries = [
    "인공지능이란 무엇인가요?",
    "딥러닝과 머신러닝의 관계는 어떻게 되나요?",
    "컴퓨터가 이미지를 이해하는 방법은?"
]

# 각 쿼리에 대해 가장 유사한 문서 찾기
for query in queries:
    most_similar_doc, similarity = find_most_similar(
        query, 
        document_embeddings_openai, 
        embeddings_model=embeddings_openai
        )
    print(f"쿼리: {query}")
    print(f"가장 유사한 문서: {most_similar_doc}")
    print(f"유사도: {similarity:.4f}")
    print()
    

쿼리: 인공지능이란 무엇인가요?
가장 유사한 문서: 인공지능은 컴퓨터 과학의 한 분야입니다.
유사도: 0.7114

쿼리: 딥러닝과 머신러닝의 관계는 어떻게 되나요?
가장 유사한 문서: 딥러닝은 머신러닝의 한 종류입니다.
유사도: 0.6825

쿼리: 컴퓨터가 이미지를 이해하는 방법은?
가장 유사한 문서: 컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.
유사도: 0.7054



### 2. **Huggingface**

- LangChain에서 오픈소스 기반의 대표적인 임베딩 모델

- 주요 특징:
    1. 로컬 환경에서 실행 가능
    2. 다양한 사전학습 모델 지원
    3. 커스텀 모델 학습 및 적용 가능
    4. 무료 사용 가능 (API 비용 없음)

- 사용시 주의사항:
    1. 로컬 컴퓨팅 자원 필요 (CPU/GPU)
    2. 초기 모델 다운로드 시간 소요
    3. 메모리 사용량 고려 필요
    4. transformers 라이브러리 설치 필요

- 임베딩 벡터 특성:
    1. 모델별로 다양한 차원 제공 (128 ~ 1024)
    2. sentence-transformers 기반 구현
    3. BERT 계열 모델 구조 사용
    4. 코사인 유사도 기반 검색 최적화

- 대표적인 임베딩 모델:

    | 모델명 (Hugging Face ID) | 차원 | 언어 | 특징 및 추천 용도 |
    | :--- | :---: | :---: | :--- |
    | **`all-MiniLM-L6-v2`** | 384 | **영어** | **[영어 표준/경량]** 매우 빠르고 메모리 효율이 좋음. 영어 전용 검색/분류 작업의 입문용 모델. |
    | **`all-mpnet-base-v2`** | 768 | **영어** | **[영어 고성능]** MiniLM보다 느리지만, 문장의 뉘앙스를 가장 정확하게 포착함. 영어권 RAG의 표준. |
    | **`paraphrase-multilingual-MiniLM-L12-v2`** | 384 | 다국어 | **[다국어 경량]** 한국어를 포함한 50개국어 지원. 속도가 빨라 실시간 서비스에 적합. |
    | **`intfloat/multilingual-e5-large`** | 1024 | 다국어 | **[다국어 고성능]** 다국어 벤치마크 상위권 모델. (사용 시 `query:`, `passage:` 접두어 필요) |
    | **`BAAI/bge-m3`** | 1024 | 다국어 | **[한국어 최적]** 한국어 처리 성능이 매우 뛰어나며, 긴 문장(8192 토큰)도 처리 가능. |


`(1) embedding 모델`

- langchain_huggingface 설치 필요

In [15]:
from langchain_huggingface import HuggingFaceEmbeddings  

# Hugging Face의 임베딩 모델 생성
embeddings_gemma = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",          # 사용할 모델 이름 - 구글의 경량 임베딩 모델
    # model_kwargs={'device': 'cuda'}  # GPU 사용시
    # model_kwargs={'device': 'mps'}   # Mac M1 사용시
)

# 임베딩 객체 출력
embeddings_gemma

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 32541.73it/s]


HuggingFaceEmbeddings(model_name='BAAI/bge-m3', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [17]:
import torch
print("CUDA 가용한가요?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("사용 중인 GPU:", torch.cuda.get_device_name(0))

CUDA 가용한가요?: False


`(2) embed_documents 사용`

In [16]:
# 문서 임베딩
document_embeddings_gemma = embeddings_gemma.embed_documents(documents)

# 임베딩 결과 출력
print(f"임베딩 벡터의 개수: {len(document_embeddings_gemma)}")
print(f"임베딩 벡터의 차원: {len(document_embeddings_gemma[0])}")
print(document_embeddings_gemma[0])

임베딩 벡터의 개수: 5
임베딩 벡터의 차원: 1024
[-0.039414480328559875, 0.008764872327446938, -0.012681577354669571, 0.0024531777016818523, -0.008944753557443619, -0.007383654825389385, -0.0053773666732013226, -0.009055884554982185, 0.03291522338986397, 0.006045493297278881, -0.027012905105948448, -0.02774086594581604, 0.00044412061106413603, 0.030136531218886375, 0.017242833971977234, 0.0170903280377388, 0.025524886325001717, -0.021856091916561127, -0.011341285891830921, -0.057022612541913986, -0.0003017036069650203, 0.013543075881898403, -0.00745011493563652, 0.01857437938451767, 0.0028946585953235626, 0.008630624040961266, -0.0007445244118571281, -0.028904100880026817, 0.020727822557091713, -0.02050059288740158, 0.008069886825978756, -0.02675417810678482, 0.003963073715567589, -0.016303930431604385, -0.07406219840049744, -0.03365038335323334, -0.023871466517448425, -0.034550078213214874, -0.034785956144332886, 0.005482956767082214, -0.0500335656106472, -0.002803527284413576, -0.02314688451588154, -0

`(3) embed_query 사용`

In [18]:
embedded_query = embeddings_gemma.embed_query("인공지능이란 무엇인가요?")

# 쿼리 임베딩 결과 출력
print(f"쿼리 임베딩 벡터의 차원: {len(embedded_query)}")
print(embedded_query)

쿼리 임베딩 벡터의 차원: 1024
[-0.037039078772068024, -0.004837995395064354, 0.002937317593023181, -0.015514637343585491, -0.0009441875154152513, -0.04150160402059555, -0.006574456114321947, 0.01128960307687521, 0.021614039316773415, 0.004928705282509327, -0.0203405749052763, 0.016905223950743675, -0.01287414412945509, 0.005518912337720394, 0.014988318085670471, 0.024228785187005997, 0.007369110360741615, -0.02804991789162159, -0.014938990585505962, -0.05185188725590706, -0.006705068051815033, -0.009251533076167107, -0.01698080077767372, 0.006491474341601133, 0.0529317706823349, 0.04813732951879501, -0.008069492876529694, -0.023171789944171906, 0.01814303919672966, -0.01132810115814209, -0.004240385722368956, -0.006354686804115772, -0.0022717337124049664, 0.014329425059258938, -0.03563683480024338, -0.00815579667687416, -0.011798219755291939, -0.045424092561006546, -0.04073287919163704, 0.002213897882029414, -0.012132312171161175, 0.017896132543683052, -0.01914467290043831, -0.04192442446947098,

`(4) 유사도 기반 검색`

In [19]:
# 예제 쿼리
queries = [
    "인공지능이란 무엇인가요?",
    "딥러닝과 머신러닝의 관계는 어떻게 되나요?",
    "컴퓨터가 이미지를 이해하는 방법은?"
]

# 각 쿼리에 대해 가장 유사한 문서 찾기
for query in queries:
    most_similar_doc, similarity = find_most_similar(query, document_embeddings_gemma, embeddings_model=embeddings_gemma) 
    print(f"쿼리: {query}")
    print(f"가장 유사한 문서: {most_similar_doc}")
    print(f"유사도: {similarity:.4f}")
    print()

쿼리: 인공지능이란 무엇인가요?
가장 유사한 문서: 인공지능은 컴퓨터 과학의 한 분야입니다.
유사도: 0.7269

쿼리: 딥러닝과 머신러닝의 관계는 어떻게 되나요?
가장 유사한 문서: 딥러닝은 머신러닝의 한 종류입니다.
유사도: 0.7057

쿼리: 컴퓨터가 이미지를 이해하는 방법은?
가장 유사한 문서: 컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.
유사도: 0.6843



### 3. **Ollama (로컬 실행 최적화)**

LangChain에서 로컬 LLM 및 임베딩 모델을 가장 손쉽게 실행할 수 있는 플랫폼입니다. 외부 API 전송 없이 로컬 자원(GPU/CPU)만 사용하므로 데이터 보안과 비용 절감에 최적화되어 있습니다.

**주요 특징:**
1. **완전한 로컬 실행:** 데이터가 외부로 유출되지 않아 기업 내부(On-premise) 구축에 적합.
2. **빠른 추론 속도:** C++ 기반의 런타임과 양자화(Quantization) 기술로 최적화됨.
3. **간편한 배포:** Docker 기반으로 모델 설치 및 실행이 매우 간단함 (`ollama pull 모델명`).

**사용 시 주의사항:**
1. **서버 실행 필수:** 백그라운드에서 `ollama serve`가 실행 중이어야 함.
2. **리소스 관리:** 고성능 모델(Large) 사용 시 충분한 RAM/VRAM 필요.
3. **API 설정:** LangChain 등에서 호출 시 엔드포인트(`localhost:11434`) 확인 필요.

**대표적인 임베딩 모델 비교:**

| 모델명 (Model Tag) | 차원 | 언어 | 특징 및 추천 용도 |
| :--- | :---: | :---: | :--- |
| **`nomic-embed-text`** | 768 | 영어 | **[Ollama 표준]** 긴 문맥(8192 토큰)을 지원하며, 오픈소스 중 밸런스가 가장 우수함. |
| **`mxbai-embed-large`** | 1024 | 영어 | **[SOTA 성능]** MTEB 리더보드 상위권 모델. 검색 정확도가 매우 높음. |
| **`snowflake-arctic-embed`** | 1024 | 영어 | **[검색 최적화]** Snowflake사가 RAG 및 대규모 검색 작업에 특화하여 설계함. |
| **`bge-m3`** | 1024 | **다국어** | **[한국어 추천]** Ollama에서 사용 가능한 가장 강력한 다국어/한국어 모델. |
| **`all-minilm`** | 384 | 영어 | **[초경량]** 속도가 매우 빠르고 CPU 환경에서도 부담 없이 실행 가능. |

**임베딩 벡터 특성:**
1. **고정 차원:** 모델별로 384~1024의 고정된 벡터 차원을 가짐.
2. **자동 양자화:** 원본 모델을 4비트(Q4_0) 등으로 압축하여 메모리 사용량을 대폭 줄임.

`(1) embedding 모델`

- langchain_ollama 설치 필요

In [20]:
from langchain_ollama import OllamaEmbeddings 

# OllamaEmbeddings 모델 생성
# embeddings_ollama = OllamaEmbeddings(
#     model="nomic-embed-text",          # 사용할 모델 이름
#     base_url="http://localhost:11434"  # Ollama 서버 주소
# )
#embeddings_ollama = OllamaEmbeddings(model="bge-m3")
embeddings_ollama = OllamaEmbeddings(model="embeddinggemma")


# 임베딩 객체 출력
embeddings_ollama

OllamaEmbeddings(model='embeddinggemma', dimensions=None, validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

`(2) embed_documents 사용`

In [21]:
# 문서 컬렉션
documents = [
    "인공지능은 컴퓨터 과학의 한 분야입니다.",
    "머신러닝은 인공지능의 하위 분야입니다.",
    "딥러닝은 머신러닝의 한 종류입니다.",
    "자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.",
    "컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다."
]

# 문서 임베딩
document_embeddings_ollama = embeddings_ollama.embed_documents(documents)

# 임베딩 결과 출력
print(f"임베딩 벡터의 개수: {len(document_embeddings_ollama)}")
print(f"임베딩 벡터의 차원: {len(document_embeddings_ollama[0])}")
print(document_embeddings_ollama[0])

임베딩 벡터의 개수: 5
임베딩 벡터의 차원: 768
[-0.015139604, 0.0025080813, 0.026455438, -0.03936301, 0.017339895, 0.025682883, 0.080200516, 0.025314216, 0.03688383, -0.06486216, 0.012677273, -0.024210405, 0.0105433995, -0.044417128, 0.064744405, -0.024615005, 0.0018832552, 0.04296191, -0.053310513, 0.004040536, 0.036572725, -0.0064939857, -0.062784426, -0.026538266, 0.022826793, 0.006144003, 0.005915941, -0.0320008, 0.020836197, 0.014261853, 0.022260284, -0.043133102, 0.033592205, 0.019886028, 0.019191943, 0.030565329, 0.027202748, -0.034497924, 0.018760888, -0.021154724, -0.023525154, 0.052828446, -0.01262015, 0.05914487, 0.05882816, -0.017139286, -0.04941169, -0.077339545, 0.025167173, -0.07939149, 0.003042151, -0.068163164, -0.04738519, 0.035267454, -0.048902117, -0.03671051, -0.012312264, -0.01180197, -0.04444489, -0.0015079553, -0.042776227, 0.0068242503, -0.033661924, -0.002043754, 0.086509764, -0.03620764, 0.019434817, -0.009070895, 0.0030294384, 0.11865004, 0.079337925, -0.046169892, -0.024944

`(3) embed_query 사용`

In [22]:
embedded_query = embeddings_ollama.embed_query("인공지능이란 무엇인가요?")

# 쿼리 임베딩 결과 출력
print(f"쿼리 임베딩 벡터의 차원: {len(embedded_query)}")
print(embedded_query)

쿼리 임베딩 벡터의 차원: 768
[-0.045380734, -0.020966075, 0.029104933, -0.004983477, 0.009401518, 0.057456244, 0.051829066, 0.040092185, 0.059799656, 0.023100855, -0.021187639, -0.0077385167, 0.014072221, -0.013548244, 0.06927665, -0.05156498, 0.021390215, -0.011006411, -0.106099956, 0.021674596, -0.0101300925, -0.02309053, -0.0419618, -0.002286789, 0.027205434, 0.0038935652, 0.01579333, -0.06842342, -0.005903444, 0.010907213, 0.03226686, -0.07536957, 0.027020464, -0.010622569, 0.040005606, 0.047188845, 0.021189706, -0.015618355, -0.032879118, -0.007934205, -0.0070347446, 0.020681743, -0.017283913, 0.045291234, 0.00970471, -0.023320802, -0.08048198, -0.030822821, 0.040895104, -0.047454752, 0.019899214, -0.034309477, -0.08022102, 0.025515065, -0.009101713, 0.005705067, 0.047180038, -0.022003554, -0.05334712, 0.013518234, -0.039465655, 0.053209383, -0.0013152021, 0.009794383, 0.082471296, 0.01737276, -0.0039118985, 0.011128127, 0.015587661, 0.1448691, 0.027284876, -0.031937595, -0.036543984, -0.02

`(4) 유사도 기반 검색`

In [23]:
# 예제 쿼리
queries = [
    "인공지능이란 무엇인가요?",
    "딥러닝과 머신러닝의 관계는 어떻게 되나요?",
    "컴퓨터가 이미지를 이해하는 방법은?"
]

# 각 쿼리에 대해 가장 유사한 문서 찾기
for query in queries:
    most_similar_doc, similarity = find_most_similar(query, document_embeddings_ollama, embeddings_model=embeddings_ollama) 
    print(f"쿼리: {query}")
    print(f"가장 유사한 문서: {most_similar_doc}")
    print(f"유사도: {similarity:.4f}")
    print()

쿼리: 인공지능이란 무엇인가요?
가장 유사한 문서: 인공지능은 컴퓨터 과학의 한 분야입니다.
유사도: 0.6788

쿼리: 딥러닝과 머신러닝의 관계는 어떻게 되나요?
가장 유사한 문서: 딥러닝은 머신러닝의 한 종류입니다.
유사도: 0.6790

쿼리: 컴퓨터가 이미지를 이해하는 방법은?
가장 유사한 문서: 컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.
유사도: 0.6733



# [실습 프로젝트]

1. OpenAI text-embedding-3-small 임베딩 모델을 초기화합니다. 
2. 임베딩 차원을 각각 512와 1536으로 구분하여 2개의 모델을 생성합니다. 
3. 아래 주어진 문장들의 임베딩을 생성합니다. (임베딩 차원이 512, 1536인 경우 각각 2개씩 생성)
   - 문장1: "인공지능은 현대 사회를 변화시키고 있다"
   - 문장2: "AI 기술이 우리의 미래를 바꾸고 있다"
4. 생성된 임베딩의 차원을 출력합니다. (임베딩 차원이 512, 1536인 경우를 각각 출력)
5. 두 문장 간의 코사인 유사도를 계산합니다. (임베딩 차원이 512, 1536인 경우를 각각 비교)
6. [추가] 임베딩 차원(512 vs 1536)에 따른 유사도 차이를 분석하고, 어떤 차원이 더 효과적인지 생각해보세요.

In [40]:
# 1. OpenAI text-embedding-3-small 임베딩 모델을 초기화합니다. 

from langchain_openai import OpenAIEmbeddings

# 2. 임베딩 차원을 각각 512와 1536으로 구분하여 2개의 모델을 생성합니다. 
embeddings_model1 = OpenAIEmbeddings(
    model="text-embedding-3-small",  # text-embedding-3-small
    dimensions=512, # 원하는 임베딩 차원 수를 지정 가능 (기본값: None)
    )

# OpenAIEmbeddings 모델 생성할 때 임베딩 차원을 지정하는 예시
embeddings_model2 = OpenAIEmbeddings(
    model="text-embedding-3-small",  # 사용할 모델 이름
    dimensions=1536, # 원하는 임베딩 차원 수를 지정 가능 (기본값: None)
    )

# 임베딩 객체 출력
embeddings_model1
embeddings_model2

# 임베딩 모델의 컨텍스트 길이 확인
embeddings_model1.embedding_ctx_length

# 임베딩 모델의 임베딩 차원 확인 
embeddings_model2.dimensions

1536

In [41]:
""" 3. 아래 주어진 문장들의 임베딩을 생성합니다. (임베딩 차원이 512, 1536인 경우 각각 2개씩 생성)
   - 문장1: "인공지능은 현대 사회를 변화시키고 있다"
   - 문장2: "AI 기술이 우리의 미래를 바꾸고 있다"
"""
sentences = ["인공지능은 현대 사회를 변화시키고 있다", "AI 기술이 우리의 미래를 바꾸고 있다", ]

embedded_query1 = embeddings_model1.embed_documents(sentences)
embedded_query2 = embeddings_model2.embed_documents(sentences)

# 4. 생성된 임베딩의 차원을 출력합니다. (임베딩 차원이 512, 1536인 경우를 각각 출력)
print(f"쿼리 임베딩 벡터의 차원: {len(embedded_query1)}")
print(f"쿼리 임베딩 벡터의 차원: {len(embedded_query2)}")
print(embedded_query1[0])
print(embedded_query1[1])
print(embedded_query2)

쿼리 임베딩 벡터의 차원: 2
쿼리 임베딩 벡터의 차원: 2
[0.01446533203125, 0.0195159912109375, 0.0178375244140625, 0.063720703125, 0.0304718017578125, -0.05767822265625, 0.00020015239715576172, 0.1422119140625, 0.003173828125, 0.02313232421875, -0.009796142578125, -0.05072021484375, -0.0164642333984375, -0.023223876953125, -0.0156097412109375, -0.090087890625, -0.0478515625, 0.006793975830078125, 0.054046630859375, -0.045928955078125, -0.07379150390625, -0.022735595703125, 0.04351806640625, -0.017242431640625, 0.03076171875, -0.0107421875, -0.03082275390625, 0.034149169921875, 0.041961669921875, 0.009033203125, -0.0063323974609375, -0.04083251953125, -0.004451751708984375, -0.07037353515625, 0.02117919921875, 0.023284912109375, -0.058135986328125, -0.0059661865234375, 0.0229644775390625, -0.0064544677734375, 0.016326904296875, -0.025299072265625, 0.057159423828125, 0.033111572265625, -0.0205078125, 0.0230255126953125, -0.039520263671875, 0.005741119384765625, 0.0197906494140625, 0.0723876953125, -0.03433227

In [42]:
#5. 두 문장 간의 코사인 유사도를 계산합니다. (임베딩 차원이 512, 1536인 경우를 각각 비교)
from langchain_community.utils.math import cosine_similarity
import numpy as np

# 512 차원 임베딩의 코사인 유사도 계산
# [embedded_query1[0]]과 [embedded_query1[1]]을 각각 2차원 리스트 형태로 전달합니다.
similarity_matrix_512 = cosine_similarity([embedded_query1[0]], [embedded_query1[1]])
similarity_512 = similarity_matrix_512[0][0]

# 1536 차원 임베딩의 코사인 유사도 계산
similarity_matrix_1536 = cosine_similarity([embedded_query2[0]], [embedded_query2[1]])
similarity_1536 = similarity_matrix_1536[0][0]

print(f"512 차원 임베딩의 코사인 유사도 : {similarity_512:.4f}")
print(f"1536 차원 임베딩의 코사인 유사도: {similarity_1536:.4f}")

512 차원 임베딩의 코사인 유사도 : 0.5538
1536 차원 임베딩의 코사인 유사도: 0.4954


In [47]:
# 6. [추가] 임베딩 차원(512 vs 1536)에 따른 유사도 차이를 분석하고, 어떤 차원이 더 효과적인지 생각해보세요.
similarity_diff = abs(similarity_1536 - similarity_512)

print("\n" + "="*50)
print("  [#6 분석 보고서] text-embedding-3-small 차원별 비교")
print("="*50)
print(f"- 512 차원 모델의 유사도 : {similarity_512:.4f}")
print(f"- 1536 차원 모델의 유사도: {similarity_1536:.4f}")
print(f"- 두 모델의 유사도 편차 : {similarity_diff:.4f}")
print("-"*50)

analysis_text = f"""
[유사도 차이 분석 및 고찰]

1. 기술적 배경 (OpenAI text-embedding-3-small)
   - 본 실험에 사용된 모델은 '마트료시카 표현 학습(Matryoshka Representation Learning)' 기법이 적용되었습니다.
   - 이는 1536차원의 핵심 정보가 앞쪽 차원에 압축되어 있어, 차원을 잘라내도 성능 손실이 적도록 설계된 구조입니다.

2. 반전된 결과 해석 (왜 512차원의 유사도가 더 높을까?)
   - 실험 결과, 512차원의 유사도(0.5538)가 원본인 1536차원(0.4954)보다 오히려 약 0.0583 더 높게 측정되었습니다.
   - 고차원(1536) 공간에서는 문장의 미세한 단어 차이나 표현 방식의 이질성까지 너무 정밀하게 포착하여 유사도가 낮게 측정될 수 있습니다.
   - 반면, 저차원(512) 공간으로 축소되는 과정에서 지엽적인 노이즈가 제거되고 문장의 '핵심적인 의미 정보' 위주로 벡터가 재구성되면서 두 동의어 문장의 유사도가 더 직관적으로 상승한 것으로 해석됩니다.

3. 결론: 어떤 차원이 더 효과적인가?
   - 본 실험(동의어 쌍 판별 태스크) 기준 효과적인 차원: [ 512 차원 ]
   - 이유: 저장 공간(Vector DB 비용)과 계산 리소스를 3배 가까이 절약하면서도, 본 문장 쌍처럼 형태는 다르지만 의미가 통하는 문맥을 더 명확하게 양의 유사도(더 높은 점수)로 잡아내기 때문에 가성비와 직관성 면에서 512차원이 훨씬 효과적입니다.
   - 확장적 관점: 다만 복잡한 다국어 처리나 미세한 도메인 지식 구분이 필요한 대규모 RAG 시스템에서는 1536차원의 정밀함이 필요할 수 있으므로, 단순 시맨틱 검색에는 512차원을 적극 권장합니다.
"""

print(analysis_text.strip())
print("="*50)


  [#6 분석 보고서] text-embedding-3-small 차원별 비교
- 512 차원 모델의 유사도 : 0.5538
- 1536 차원 모델의 유사도: 0.4954
- 두 모델의 유사도 편차 : 0.0583
--------------------------------------------------
[유사도 차이 분석 및 고찰]

1. 기술적 배경 (OpenAI text-embedding-3-small)
   - 본 실험에 사용된 모델은 '마트료시카 표현 학습(Matryoshka Representation Learning)' 기법이 적용되었습니다.
   - 이는 1536차원의 핵심 정보가 앞쪽 차원에 압축되어 있어, 차원을 잘라내도 성능 손실이 적도록 설계된 구조입니다.

2. 반전된 결과 해석 (왜 512차원의 유사도가 더 높을까?)
   - 실험 결과, 512차원의 유사도(0.5538)가 원본인 1536차원(0.4954)보다 오히려 약 0.0583 더 높게 측정되었습니다.
   - 고차원(1536) 공간에서는 문장의 미세한 단어 차이나 표현 방식의 이질성까지 너무 정밀하게 포착하여 유사도가 낮게 측정될 수 있습니다.
   - 반면, 저차원(512) 공간으로 축소되는 과정에서 지엽적인 노이즈가 제거되고 문장의 '핵심적인 의미 정보' 위주로 벡터가 재구성되면서 두 동의어 문장의 유사도가 더 직관적으로 상승한 것으로 해석됩니다.

3. 결론: 어떤 차원이 더 효과적인가?
   - 본 실험(동의어 쌍 판별 태스크) 기준 효과적인 차원: [ 512 차원 ]
   - 이유: 저장 공간(Vector DB 비용)과 계산 리소스를 3배 가까이 절약하면서도, 본 문장 쌍처럼 형태는 다르지만 의미가 통하는 문맥을 더 명확하게 양의 유사도(더 높은 점수)로 잡아내기 때문에 가성비와 직관성 면에서 512차원이 훨씬 효과적입니다.
   - 확장적 관점: 다만 복잡한 다국어 처리나 미세한 도메인 지식 구분이 필요한 대규모 RAG 시스템에서는 1536차원의 정밀함이 필요할 수 

In [44]:
# 1. OpenAI text-embedding-3-small 임베딩 모델 초기화
# 2. 임베딩 차원을 각각 512와 1536으로 구분하여 2개의 모델 생성

# 임베딩 차원 512인 모델
embeddings_512 = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=512
)

# 임베딩 차원 1536인 모델 (기본 차원)
embeddings_1536 = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1536
)

print("모델 초기화 완료")
print(f"embeddings_512 차원: {embeddings_512.dimensions}")
print(f"embeddings_1536 차원: {embeddings_1536.dimensions}")

모델 초기화 완료
embeddings_512 차원: 512
embeddings_1536 차원: 1536


In [45]:
# 3. 주어진 문장들의 임베딩 생성

# 문장 정의
sentence1 = "인공지능은 현대 사회를 변화시키고 있다"
sentence2 = "AI 기술이 우리의 미래를 바꾸고 있다"

# 임베딩 차원 512인 경우
embedding1_512 = embeddings_512.embed_query(sentence1)
embedding2_512 = embeddings_512.embed_query(sentence2)

# 임베딩 차원 1536인 경우
embedding1_1536 = embeddings_1536.embed_query(sentence1)
embedding2_1536 = embeddings_1536.embed_query(sentence2)

print("임베딩 생성 완료")

임베딩 생성 완료


In [46]:
# 4. 생성된 임베딩의 차원 출력

print("=== 임베딩 차원 확인 ===")
print(f"\n[임베딩 차원 512인 경우]")
print(f"문장1 임베딩 차원: {len(embedding1_512)}")
print(f"문장2 임베딩 차원: {len(embedding2_512)}")

print(f"\n[임베딩 차원 1536인 경우]")
print(f"문장1 임베딩 차원: {len(embedding1_1536)}")
print(f"문장2 임베딩 차원: {len(embedding2_1536)}")

=== 임베딩 차원 확인 ===

[임베딩 차원 512인 경우]
문장1 임베딩 차원: 512
문장2 임베딩 차원: 512

[임베딩 차원 1536인 경우]
문장1 임베딩 차원: 1536
문장2 임베딩 차원: 1536


In [48]:
# 5. 두 문장 간의 코사인 유사도 계산

from langchain_community.utils.math import cosine_similarity

# 임베딩 차원 512인 경우의 유사도
similarity_512 = cosine_similarity([embedding1_512], [embedding2_512])[0][0]

# 임베딩 차원 1536인 경우의 유사도
similarity_1536 = cosine_similarity([embedding1_1536], [embedding2_1536])[0][0]

print("=== 코사인 유사도 비교 ===")
print(f"\n문장1: {sentence1}")
print(f"문장2: {sentence2}")
print(f"\n[임베딩 차원 512] 코사인 유사도: {similarity_512:.4f}")
print(f"[임베딩 차원 1536] 코사인 유사도: {similarity_1536:.4f}")
print(f"\n유사도 차이: {abs(similarity_512 - similarity_1536):.4f}")

=== 코사인 유사도 비교 ===

문장1: 인공지능은 현대 사회를 변화시키고 있다
문장2: AI 기술이 우리의 미래를 바꾸고 있다

[임베딩 차원 512] 코사인 유사도: 0.5538
[임베딩 차원 1536] 코사인 유사도: 0.4955

유사도 차이: 0.0583
